In [ ]:
import torch
import math


class LinearLayer:
    def __init__(self,in_features,out_features,bias=False):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.has_bias= bias
        if bias:
            self.bias = torch.zeros(out_features)
        else:
            self.bias = None

    def forward(self,x):
        self.x  = x
        out = self.x @ self.weights.T
        if self.has_bias:
            out = out+self.bias
        
        return out


    def backward(self,grad_out):
        x_shape = self.x.shape
        x_flat = self.x.flatten(0,-2)
        grad_flat = grad_out.flatten(0,-2)

        grad_inputs = grad_flat @ self.weights
        self.weights.grad = grad_flat.T @ x_flat
        if self.has_bias:
            self.bias.grad = grad_flat.sum(dim=0)

        grad_inputs = grad_inputs.reshape(x_shape)
        return grad_inputs

class softmax:

    def forward(self,scores):
        max_score = torch.max(scores,dim=-1,keepdim=True).values
        scores_exp = torch.exp(scores-max_score)
        self.scores_sum = scores_exp.sum(dim=-1,keepdim=True)
        self.out = scores_exp/self.scores_sum
        return self.out

    
    def backward(self,grad_attn):

        sum_term = (grad_attn *self.out ).sum(dim=-1, keepdim=True)
        grad_scores = self.out * (grad_out - sum_term)

        return grad_scores
         

class CausalSelfAttention:
    def __init__(self,num_dims,num_heads):
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.head_dims = num_dims // num_heads

        self.w_q = LinearLayer(num_dims,num_dims,bias=False)
        self.w_k = LinearLayer(num_dims,num_dims,bias=False)
        self.w_v = LinearLayer(num_dims,num_dims,bias=False)

        self.proj_out = LinearLayer(num_dims,num_dims,bias=True)
        self.softmax = softmax()

    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q.forward(x)
        K = self.w_k.forward(x)
        V = self.w_v.forward(x)

        Q = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        self.Q = Q
        self.K = K
        self.V = V
        
        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T,T,dtype=torch.bool,device=scores.device),diagonal=1)

        scores = scores.masked_fill(masks,float("-inf"))

        self.attn_scores = self.softmax.forward(scores)
        self.out = self.attn_scores @ self.V
        self.out = self.out.transpose(1,2).contiguous().view(B,T,D)
        self.out = self.proj_out(self.out)

        return self.out

    def backward(self,grad_out):

        grad_proj = self.proj_out.backward(grad_out)
        grad_proj = grad_proj.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        
        grad_attn = grad_proj @ self.V.transpose(-2,-1)
        grad_V = self.attn_scores.transpose(-2,-1) @ grad_proj

        grad_scores = self.softmax.backward(grad_attn)   

        scale = 1.0 / math.sqrt(self.head_dims)
        grad_scores = grad_scores * scale
        
        grad_q = grad_scores @ self.K
        grad_k = grad_scores.transpose(-2,-1) @ self.Q
        

        grad_Q = grad_Q.transpose(1, 2).contiguous().view(B, T, D)
        grad_K = grad_K.transpose(1, 2).contiguous().view(B, T, D)
        grad_V = grad_V.transpose(1, 2).contiguous().view(B, T, D)

        grad_x_q = self.w_q.backward(grad_Q)
        grad_x_k = self.w_k.backward(grad_K)
        grad_x_v = self.w_v.backward(grad_V)

        grad_x = grad_x_q+grad_x_k+grad_x_v

        return grad_x

    def parameters(self):
        return [self.w_q.weights, self.w_k.weights, self.w_v.weights, self.proj_out.weights, self.proj_out.bias]
        

        
    

        